# 03b · Hospital activity: admissions, discharges, transfers, and what changes in SQL

**Who this is for:** you, after notebook 03.
**You should already know:** the truth generator (notebook 03) and the Azure SQL tables (notebook 01).

**By the end you will be able to:**
- explain why a realistic ward must keep **moving**: people go home, beds are refilled, patients transfer
- read a hospital schedule: lengths of stay, bed idle time, wearable devices passed between patients
- see how **escalations** end stays early in the actual hospital records
- look at the Azure SQL tables **at any moment**, and work out exactly which rows an **incremental ingestion** run (Lakeflow Connect, query-based) would fetch
- explain what that approach can't see compared with full CDC

**Outline**
1. Setup
2. The empty-ward problem
3. How the schedule works
4. 14 days of hospital activity
5. Devices are passed from patient to patient
6. Adding the truth: escalations end stays
7. The Azure SQL tables at a moment in time
8. Incremental ingestion: what changed since the last run?
9. What a query-based connector can't see
10. Exercise
11. Pitfalls and next steps

Code used: `generator/activity.py`, `generator/truth.py` (tests in `tests/test_activity.py`).

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from generator import activity, hospital, synthea, truth


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for folder in [start, *start.parents]:
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found")


ROOT = find_repo_root()
RUN = synthea.SyntheaRun()
csv_dir = synthea.run_synthea(RUN, ROOT / "data" / "synthea" / f"p{RUN.population}_s{RUN.seed}")
profiles = synthea.build_profiles(
    synthea.load_tables(csv_dir), pd.Timestamp(RUN.reference_date, tz="UTC")
)
inpatients = hospital.admit_inpatients(profiles)
START = hospital.SIM_START
DAYS = 14
print(len(inpatients), "inpatients at the start |", len(profiles), "people who could be admitted")

## 2. The empty-ward problem

In notebook 03 the ward was **static**: the same 300 patients for the whole run. Escalated patients left, and nobody replaced them.
Over a 14-day baseline that slowly empties the hospital, and real wards never look like that.

In [ ]:
static = truth.simulate_ward(inpatients, START, hours=24 * DAYS, seed=42)
static_census = (
    static.readings.assign(hour=static.readings["ts"].dt.floor("h"))
    .groupby("hour")["device_id"]
    .nunique()
)
ax = static_census.plot(figsize=(10, 3), title="Static ward: patients sending vitals each hour")
ax.set_ylabel("patients");

## 3. How the schedule works

`activity.schedule_activity` is a small **event-driven simulation**. It keeps a queue of future events (discharge, admission, transfer) and processes them in time order:

| Event | What happens |
|---|---|
| **Discharge** | At the end of the planned stay the patient goes home. Their bed and device become free. |
| **Admission** | After the bed has been empty for a while (cleaning, waiting), a new patient is admitted. They're drawn from the population, weighted toward older and sicker people (notebook 01) **and** toward patients who suit that unit (COPD → respiratory). |
| **Transfer** | ~15% of stays include one move to another unit type at the same site, if a bed is free there |

Lengths of stay follow a **log-normal** distribution (most stays 2–7 days, a long tail of longer ones). The settings:

In [ ]:
activity.ActivityConfig()

Devices are **wearables**: a patient gets a free monitor at admission, keeps it through any transfer, and hands it back at discharge.
That's why vitals messages carry only `device_id`, and why the platform needs the `device_assignments` table to know whose vitals they are.

## 4. 14 days of hospital activity

In [ ]:
%%time
act = activity.schedule_activity(profiles, inpatients, START, hours=24 * DAYS, seed=42)
print(f"stays: {len(act.encounters):,} | transfers: {len(act.transfers)}")
print(f"different people admitted: {len(act.patients):,}")

In [ ]:
census = activity.census(act)
enc = act.encounters
admissions = enc.loc[enc["admit_ts"] >= START, "admit_ts"].dt.floor("D").value_counts().sort_index()
discharges = enc["discharge_ts"].dropna().dt.floor("D").value_counts().sort_index()
stay_days = ((enc["discharge_ts"] - enc["admit_ts"]).dt.total_seconds() / 86400).dropna()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))
census.plot(ax=axes[0], title="Patients in hospital")
axes[0].axhline(360, ls="--", color="grey", lw=1)
pd.DataFrame({"admissions": admissions, "discharges": discharges}).plot.bar(
    ax=axes[1], title="Per day", width=0.8
)
axes[1].set_xticklabels([d.strftime("%d %b") for d in admissions.index], rotation=90, fontsize=7)
stay_days.plot.hist(
    bins=30, ax=axes[2], title=f"Length of stay (median {stay_days.median():.1f} days)"
)
plt.tight_layout()

**Interpretation:** the census hovers around 300 of the 360 beds (grey line), about 83% full, with roughly 60 admissions and 60 discharges a day. That's a living hospital.

In [ ]:
act.transfers.head()

## 5. Devices are passed from patient to patient

Each bar is one patient wearing the device. Gaps are times the device sat unused between patients.

In [ ]:
dev = act.device_assignments
some = dev["device_id"].drop_duplicates().sort_values().iloc[:12]
fig, ax = plt.subplots(figsize=(11, 4))
for y, device in enumerate(some):
    for _, a in dev[dev["device_id"] == device].iterrows():
        left = max(a["start_ts"], START)
        right = a["end_ts"] if pd.notna(a["end_ts"]) else act.end
        ax.barh(y, right - left, left=left, height=0.6)
ax.set_yticks(range(len(some)), some, fontsize=8)
ax.set_title("Device usage: each coloured bar is one patient's stay");

## 6. Adding the truth: escalations end stays

`truth.simulate_ward(..., activity=act)` now runs on this moving hospital. Each device is a **slot**, and the patient wearing it changes over time. Vitals, the hidden rule and trajectories are computed for whoever is wearing it.
When a patient escalates, their stay ends at the event (they've left for ICU or a higher-care area), and any transfer planned after that never happens.

In [ ]:
%%time
ward = truth.simulate_ward(inpatients, START, hours=24 * DAYS, seed=42, activity=act)
print(f"{len(ward.readings):,} readings | {len(ward.truth.outcomes)} escalations")
actual = ward.activity  # the real hospital records: escalations applied
actual.encounters["disposition"].value_counts()

In [ ]:
patient_days = len(ward.readings) / 288
print(f"Escalations per 100 patient-days: {100 * len(ward.truth.outcomes) / patient_days:.1f}")
actual.encounters.query("disposition == 'escalated'")[
    ["encounter_id", "site_id", "unit_id", "admit_ts", "planned_discharge_ts", "discharge_ts"]
].head()

Notice `discharge_ts` is **earlier** than `planned_discharge_ts` for escalated stays. The plan said "home on Thursday", but the patient deteriorated on Tuesday.
The bed then stays empty until its next planned admission.

## 7. The Azure SQL tables at a moment in time

In Phase 1 these records live in **Azure SQL**. `activity.sql_snapshot` shows the `encounters` and `device_assignments` tables exactly as they'd look at any moment, including each row's **`last_updated`**: the time of its latest change.

We use the **actual** records (`ward.activity`), so escalated patients appear as discharged at the moment they left.

Here's the hospital on the morning of **day 3**, when drift episode 1 runs (Plan.md §6.4):

In [ ]:
DAY3 = START + pd.Timedelta(days=3, hours=10)
DAY8 = START + pd.Timedelta(days=8, hours=10)
snap3 = activity.sql_snapshot(actual, DAY3)
print(snap3["encounters"]["status"].value_counts().to_dict())
snap3["encounters"].sort_values("last_updated").tail()

How `last_updated` is set:

| Change | SQL statement | `last_updated` becomes |
|---|---|---|
| Admission | `INSERT` into encounters and device_assignments | admission time |
| Transfer | `UPDATE encounters SET unit_id, bed_id` | transfer time |
| Discharge | `UPDATE encounters SET discharge_ts, status` and `UPDATE device_assignments SET end_ts` | discharge time |

## 8. Incremental ingestion: what changed since the last run?

Lakeflow Connect's query-based connector remembers the highest `last_updated` it saw (its **cursor**). The next run asks only for rows changed after it:

```sql
SELECT * FROM dbo.encounters WHERE last_updated > @cursor
```

Suppose the day-3 episode ingested everything, and the next run is the **day-8** episode:

In [ ]:
changes = activity.changes_between(actual, DAY3, DAY8)
snap8 = activity.sql_snapshot(actual, DAY8)
for table in ["encounters", "device_assignments"]:
    total = len(snap8[table])
    fetched = len(changes[table])
    print(
        f"{table:<20} rows: {total:>5,} | day-8 run fetches: {fetched:>4,} ({fetched / total:.0%})"
    )

In [ ]:
c = changes["encounters"]
inserted = c["admit_ts"] > DAY3
kinds = pd.Series(
    {
        "new admissions (INSERT)": int(inserted.sum()),
        "discharged since day 3 (UPDATE)": int((~inserted & (c["status"] == "discharged")).sum()),
        "transferred, still admitted (UPDATE)": int(
            (~inserted & (c["status"] == "admitted")).sum()
        ),
    },
    name="rows",
)
kinds

Let's watch one encounter that was **in hospital on day 3** and **discharged by day 8**. In Bronze, its day-3 row is **updated in place** (an upsert on the primary key). Silver then records the change as history (SCD2).

In [ ]:
e = c.loc[~inserted & (c["status"] == "discharged"), "encounter_id"].iloc[0]
pd.concat(
    {
        "day-3 run": snap3["encounters"].set_index("encounter_id").loc[e],
        "day-8 run": snap8["encounters"].set_index("encounter_id").loc[e],
    },
    axis=1,
)

## 9. What a query-based connector can't see

The connector sees **the state at each run**, not every step in between. Look for encounters that both **transferred** and were **discharged** between the two runs:

In [ ]:
moves = actual.transfers[actual.transfers["transfer_ts"].between(DAY3, DAY8)]
both = moves.merge(c[c["status"] == "discharged"], on="encounter_id")
print(len(both), "encounters transferred AND discharged between the two runs")
both[["encounter_id", "transfer_ts", "from_unit_id", "to_unit_id", "discharge_ts"]].head(3)

The day-8 run only sees the **final** row (the new unit, discharged). The intermediate state "transferred, still in hospital" never reaches Databricks.
With **CDC** (the gateway connector) you'd get every change as its own event. For this project that's fine:
- vitals are linked to patients through **device assignments**, which don't change on transfer
- Silver keeps history **as of each ingestion**, which is what the dashboards need
- we never **delete** rows, so nothing silently disappears

This trade-off (cheaper, no always-on gateway, but no intermediate history) is worth writing up as an ADR for the portfolio.

## 10. Exercise: how much does the run frequency matter?

The day-8 run fetched changes since day 3. Suppose instead there had also been a run on **day 7**.
1. How many `encounters` rows would the day-8 run fetch if its cursor were day 7 at 10:00?
2. How many encounters with **both** a transfer and a discharge would then be hidden between the day-7 and day-8 runs?
3. What does this tell you about run frequency vs history completeness?

In [ ]:
# Try it yourself
DAY7 = START + pd.Timedelta(days=7, hours=10)

rows_fetched = None
hidden_intermediate = None

In [ ]:
# Answer (run after trying)
DAY7 = START + pd.Timedelta(days=7, hours=10)
c7 = activity.changes_between(actual, DAY7, DAY8)["encounters"]
rows_fetched = len(c7)
moves7 = actual.transfers[actual.transfers["transfer_ts"].between(DAY7, DAY8)]
hidden_intermediate = len(moves7.merge(c7[c7["status"] == "discharged"], on="encounter_id"))
print(f"rows fetched with a day-7 cursor: {rows_fetched} (vs {len(c)} with a day-3 cursor)")
print(f"hidden transfer+discharge pairs: {hidden_intermediate} (vs {len(both)})")

More frequent runs fetch fewer rows each time and hide fewer intermediate states, but they cost more compute. For our once-per-episode design, the hidden states don't matter.

## 11. Pitfalls and next steps

**Pitfall: treating `last_updated` as the event time.** It's when the *row* last changed. A discharged encounter's `last_updated` is its discharge time, not its admission time. Use `admit_ts` / `discharge_ts` for the actual events.

**Pitfall: hard deletes.** A deleted row has no `last_updated` to find, so the connector never notices. That's why the schema uses status changes (`discharged`), never `DELETE`.

**Pitfall: forgetting device reuse.** A device ID isn't a patient. Joining vitals to patients by device **and** time window (`start_ts ≤ ts < end_ts`) is essential. Joining by device alone would mix up patients.

**Optional extension:** set `activity.ActivityConfig(transfer_prob=0.4)` and see how many intermediate states a day-3 → day-8 run would hide.

**Next notebook:** `04_drift_scenarios_preview`. The four drift scenarios applied to this moving hospital, and PSI computed by hand.